# `toa_reflectance()` — `toa_reflectance`

Converter uma banda óptica DN em reflectância TOA com ajuste solar.

**Cadeia deste exemplo:** DN óptico + MTL + QA → reflectância TOA.

Este notebook executa uma etapa sobre arquivos já preparados. **Se você tem apenas os arquivos originais Level-1, prepare as entradas indicadas na tabela antes de executar este método.**

Execute apenas os **quatro blocos de código**, de cima para baixo. Os blocos ao final, em “Consulta”, são alternativas para leitura, e não uma continuação obrigatória.


## Antes de executar: separe estes arquivos

| Campo a preencher no passo 1 | Arquivo que você deve escolher | Unidade / tipo | Como obter se ainda falta |
| --- | --- | --- | --- |
| `dn_path` | Banda original Level-1; por exemplo, `PRODUTO_B4.TIF` | DN, sem conversão prévia | GeoTIFF original B4 da cena; mudar a banda e o arquivo juntos |
| `mtl_path` | Metadados de calibração; por exemplo, `PRODUTO_MTL.txt` | Arquivo texto MTL | Arquivo original da mesma cena Level-1 |
| `qa_pixel_path` | Qualidade dos pixels; por exemplo, `PRODUTO_QA_PIXEL.TIF` | Bits inteiros, sem calibração | Arquivo original QA_PIXEL da mesma cena e na mesma grade |
| `qa_radsat_path` | Saturação/qualidade geométrica; por exemplo, `PRODUTO_QA_RADSAT.TIF` | Bits inteiros, sem calibração | Arquivo original QA_RADSAT da mesma cena e na mesma grade |

Todos os GeoTIFFs devem representar a mesma cena e ter dimensões, CRS e transformação idênticos. Para recortes, use a mesma janela nas QA. As unidades são verificadas pelo usuário: a leitura do arquivo não comprova sua grandeza física.

**Configuração desta execução:** Execute B4 com `banda=4` e B5 com `banda=5`, em execuções separadas. O ajuste solar usa o MTL.

**Ambiente:** instale `python -m pip install -e ".[examples]"` na pasta da biblioteca e selecione o kernel desse ambiente no VS Code/Jupyter.


## Passo 1 — Preencher caminhos e escolher a saída

Substitua cada `CAMINHO_...` pelo caminho completo do seu arquivo, mantendo `Path(...)`. No Windows, escreva, por exemplo, `Path(r"C:\dados\radiancia_B4.tif")`. Preencha todos os campos da tabela, inclusive as duas QA.

Execute B4 com `banda=4` e B5 com `banda=5`, em execuções separadas. O ajuste solar usa o MTL.

Ajuste `saida` para a pasta desejada. Caminhos relativos partem da pasta de trabalho do kernel; a célula imprime o caminho absoluto. Em nova execução com o mesmo destino, o GeoTIFF é substituído.


In [ ]:
from pathlib import Path
import numpy as np
import rasterio

# EDITE OS CAMINHOS E AS CONFIGURAÇÕES ABAIXO; mantenha o método deste notebook.
from pylstemp import read_quality_mask
from pylstemp import toa_reflectance, read_mtl

dn_path = Path("CAMINHO_DA_BANDA_DN.tif")

banda = 4
mtl_path = Path("CAMINHO_DO_MTL.txt")
qa_pixel_path = Path("CAMINHO_DO_QA_PIXEL.tif")
qa_radsat_path = Path("CAMINHO_DO_QA_RADSAT.tif")
saida = Path("resultados/toa_reflectance") / f"B{banda}"

# Confira os arquivos antes de passar ao passo 2.
entradas_obrigatorias = {
    "dn_path": dn_path,
    "mtl_path": mtl_path,
    "qa_pixel_path": qa_pixel_path,
    "qa_radsat_path": qa_radsat_path,
}
pendentes = [f"{nome}: {caminho}" for nome, caminho in entradas_obrigatorias.items()
             if not caminho.is_file()]
if pendentes:
    raise FileNotFoundError("Preencha os caminhos de entrada com arquivos existentes:\n"
                            + "\n".join(pendentes))
print("Arquivos encontrados. Próximo passo: ler os dados e aplicar QA.")
print("Pasta de saída:", saida.resolve())


## Passo 2 — Ler as entradas, conferir a grade e aplicar QA

Execute esta célula depois de confirmar os caminhos. Ela carrega os dados, confere seu alinhamento, lê QA_PIXEL/QA_RADSAT com `read_quality_mask()` e combina QA com nodata em `mascara`. `True` indica pixel excluído. A leitura não recalibra dados prontos.

Água é preservada na política QA deste exemplo (`exclude_water=False`). A célula imprime as dimensões e as contagens de exclusão.


In [ ]:
arquivos = {
    "dn": dn_path,
}
dados = {}
grade = None
for nome, caminho in arquivos.items():
    with rasterio.open(caminho) as src:
        grade_atual = (src.shape, src.crs, src.transform)
        if grade is None:
            grade = grade_atual
            perfil = src.profile.copy()
        elif grade_atual != grade:
            raise ValueError(f"A imagem {nome} não está na mesma grade das outras entradas.")
        dados[nome] = src.read(1, masked=True).astype(float).filled(np.nan)
dn = dados["dn"]
mascara = ~np.isfinite(dn)
metadata = read_mtl(mtl_path)
# Ler QA na grade da primeira imagem e combinar com nodata antes do método.
mascara_qa = read_quality_mask(
    qa_pixel_path, qa_radsat_path,
    reference_path=next(iter(arquivos.values())),
    exclude_water=False,
)
mascara |= mascara_qa

print("Dimensões:", mascara.shape)
print("Pixels excluídos por QA:", int(mascara_qa.sum()))
print("Pixels excluídos por QA ou nodata:", int(mascara.sum()), "de", mascara.size)


## Passo 3 — Calcular o produto

Execute a chamada abaixo. As variáveis são matrizes carregadas no passo 2; os caminhos de arquivo foram usados no passo 1. `mask=mascara` aplica a seleção QA/nodata ao cálculo.


In [ ]:
resultado = toa_reflectance(
    band=banda,
    dn=dn,
    metadata=metadata,
    sun_elevation=None,
    mask=mascara,
)
imagem = resultado

## Passo 4 — Salvar e usar a saída

Execute esta célula para gravar o GeoTIFF em `saida`, preservando a grade. Confira o caminho absoluto e a quantidade de pixels válidos impressos. Pixels inválidos são gravados como NaN.

**Próximo uso:** Fornecer a DOS ou COST, ou a uma cadeia que explicitamente use reflectância TOA. O produto ainda não é reflectância superficial.


In [ ]:
saida.mkdir(parents=True, exist_ok=True)
perfil.update(driver="GTiff", dtype="float32", count=1, nodata=np.nan, compress="deflate")
with rasterio.open(saida / "toa_reflectance.tif", "w", **perfil) as dst:
    dst.write(np.asarray(imagem, dtype="float32"), 1)
    dst.update_tags(units='dimensionless', method='toa_reflectance')
print("Pixels válidos:", int(np.isfinite(imagem).sum()), "de", imagem.size)
print("Saída:", saida.resolve())

## Como conferir a execução

Abra o GeoTIFF salvo em um SIG e confira a posição, a unidade e a banda informadas acima. Compare a contagem de pixels válidos com a área disponível: QA pode excluir pixels, e os critérios do próprio método podem produzir lacunas adicionais.

| Se acontecer | O que conferir |
| --- | --- |
| `FileNotFoundError` no passo 1 | Preencha os caminhos listados no erro, inclusive MTL quando solicitado e ambas as QA. |
| Erro de grade no passo 2 | Prepare arquivos com a mesma projeção, transformação, dimensões e recorte. Não redimensione as QA por interpolação contínua. |
| Erro de domínio no passo 3 | Confira a grandeza/unidade e os requisitos do método. |
| Saída com NaN | Confira as contagens QA/nodata e o suporte do método; NaN não deve ser substituído por temperatura inventada. |

[Voltar ao índice e à ordem de execução](README.md).


## Consulta — opções e detalhes do método

O roteiro executável termina acima. Leia os tópicos abaixo para adaptar a entrada ou consultar parâmetros; não execute todas as alternativas em sequência sobre a mesma saída.


### Entradas da função e opções

### Entradas do caminho principal

| Entrada da função | O que fornecer | Unidade / domínio |
|---|---|---|
| `dn` | Imagem original Level-1 da banda escolhida | DN |

**Além da imagem:** número da banda (ópticas 1 a 9) e `metadata`, lido com `read_mtl()` do MTL Level-1 da mesma cena. DN de Level-2 SR/ST não é uma entrada equivalente.

Todas as imagens usadas juntas devem ter a mesma grade: projeção, resolução, posição dos pixels e dimensões. As variáveis da chamada são arrays de pixels; não são caminhos de arquivo.

### O que é opcional

- **`mask`:** máscara booleana alinhada: True exclui o pixel e False permite avaliar. Opcional na API; neste exemplo é preenchida no passo 2 com QA e nodata. Não passar QA_PIXEL bruta.

### Alternativas de uso

Escolha a alternativa que corresponde aos dados disponíveis; não execute todas as chamadas em sequência sobre a mesma saída.

### 5.1. Fornecer elevação solar explicitamente

sun_elevation pode ser um número ou raster alinhado em graus, em (0,90]. None usa SUN_ELEVATION do MTL. A banda SZA Landsat é zênite em centésimos de grau: converter com `90-SZA/100` e mascarar seus valores inválidos. Não fornecer SZA diretamente como elevação.

```python
imagem = toa_reflectance(
    band=banda,
    dn=dn,
    metadata=metadata,
    sun_elevation=elevacao_solar,
    mask=mascara,
)
```

### Referência completa dos parâmetros

### Caminho principal

A chamada completa e a tabela abaixo seguem a mesma ordem.

```python
resultado = toa_reflectance(
    band=banda,
    dn=dn,
    metadata=metadata,
    sun_elevation=None,
    mask=mascara,
)
```

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `band` | Obrigatório | Sem padrão | Inteiro, sem unidade | Bandas ópticas 1 a 9; usar o número correspondente ao GeoTIFF lido e aos coeficientes MTL. |
| `dn` | Obrigatório | Sem padrão | DN | Array original da banda Level-1. Não é radiância nem reflectância pronta; valores <=0 ou não finitos produzem NaN. |
| `metadata` | Obrigatório | Sem padrão | Sem unidade (dicionário) | MTL da própria cena, lido com read_mtl(); PROCESSING_LEVEL deve começar por L1. |
| `sun_elevation` | Opcional | `None` → elevação do MTL | Graus, (0,90] | Elevação solar escalar ou array na mesma grade do DN. None usa SUN_ELEVATION; raster de ângulo inválido mascara seus pixels. |
| `mask` | Opcional na API; fornecido neste exemplo | `None` | Booleano | Máscara de exclusão alinhada: True rejeita, False permite avaliar. O passo 2 lê ambas as QA e combina seus bits com nodata. Água segue a escolha exclude_water explícita neste notebook. Não fornecer QA bruta; a função pode acrescentar rejeições por domínio. |

Os valores escolhidos no exemplo podem diferir do padrão da API; a coluna de padrão descreve a função. As máscaras e as validações de domínio podem produzir NaN, sem preenchimento automático.

### Aplicabilidade e limites

A correção solar usa a elevação do MTL quando sun_elevation=None. Reflectância TOA é adimensional, ainda contém atmosfera e não equivale a SR. Valores fora de [0,1] não são truncados. Não aplicar a correção solar duas vezes.

Execução e consistência numérica não demonstram acurácia independente em campo. A parametrização deve ser adequada à banda, sensor, superfície e condições da aplicação.

[USGS — equações de calibração Level-1](https://www.usgs.gov/landsat-missions/using-usgs-landsat-level-1-data-product).

[Voltar ao índice dos exemplos](README.md).